# 0. Radon Thesis

Matthew Thompson, fall 2026, EWU. 

## 0.1 Abstract

In this paper we investigate the basic idea of the Radon transform and its use in tomographic reconstruction. We consider the filtered back projection method of inversion, deriving it from the Fourier slice theorem. We analyze the numerical attributes of various common choices of filter kernel. We construct a filter kernel which is a signed Gaussian mixture by using machine learning. We train a nerual network which plays the role of a filter kernel. Finally, we give a brief discussion of generalized Radon transforms and the theory of Radon transforms.  

## 0.2 Contents

1. Definitions and introduction
   1. Introduction
   2. Radon Transform
   3. Filtered Back Projection
2. Theory of Radon transform in R^2
   1. Fourier slice theorem
   2. Mapping of schwartz space, plancherel's formula
   3. Derivation of FBP
   4. FBP range, injectivity results (proof sketches only)
   5. A brief mention of sampling theory
3. Techniques in Parallel beam X-Ray tomography
   1. Literature review, list of filters
   2. Noise
   3. Numerical analysis
4. Learning of filter kernel
   1. Difference of gaussians filter window, least squares regression
   2. CNN filter window
   3. In-between approach, assessing noise and selecting from a family of filters
5. (Can omit) A brief discussion of further topics
   1. Radon transform in higher dimensions
   2. Reconstruction of probability distributions, cramer-wold theorem
   3. Support theorem
   4. Distriubtions
   5. Generalized Radon transform, a few examples
   6. more on the Radon transform (homogeneous spaces, double fibration)

# 1. Definitions, basic ideas, and introduction

## 1.1 Introduction

Inverse problems involve solving $f(x) = y$ for $x$, given $f$ and $y$. We can think of this as reconstructing $x$ from $y=f(x)$, or as finding the inverse function $f^{-1}$ and applying it to $f^{-1}(y) = x$. 

Computed Tomography is such an inverse problem. 

#### Physical scenario of Tomography

Radon inversion is used in medical imaging to reconstruct images from sensor data. X-rays passing through matter experience attenuation according to known laws: $I = I_0 e^{-Al}$. Thus we have a way to physically measure the line integrals in the Radon transform. Collecting sensor data from all angles and a range of displacements gives the full radon transform data. The transform of the density function in the Radon domain is called a sinogram. 

(In real applications, fan-beam and limited angle tomography is more common. But this paper will focus on parallel beam x-ray.)

## 1.2 Radon Transform

####  Radon transform

The Radon transform maps a function $f$ defined on $\mathbb{R}^n$ to a function defined over the set of all $n-1$ dimensional hyperplanes in $\mathbb{R}^n$. 

\begin{align*} \mathcal{R}f (r, \alpha) = \int_{\langle x, \alpha \rangle = r} f(x) dx \end{align*}




For the case of $\mathbb{R}^2$, given a function $f: \mathbb{R}^2 \to \mathbb{R}$, and the lines $\Gamma(r, a) = \{x, y \in \mathbb{R}^2 | r = x \cos(a) + y \sin(a)\}$, we define the radon transform as 

\begin{align*} \mathcal{R}f (r, a) = \int_{-\infty}^{\infty}    f(r\cos(a) - t \sin(a), r\sin(a) + t \cos(a)  )     dt. \end{align*}






We say a point $(x, y)$  is incident with the line $\Gamma(r, a)$ if they "touch", that is if $(x,y) = (r\cos(a) - t \sin(a), r\sin(a) + t \cos(a)$ for some $t$. We can capture this idea of incidence by writing the Radon transform in terms of the Dirac delta function.

\begin{align*} \mathcal{R}f (r, a) = \int \int_{\mathbb{R}^2} f(x, y) \delta (r - x \cos(a) - y \sin(a)) dx dy \end{align*}

We can think of $\mathcal{R}f$ as a function on the tangent bundle to $S^1$, since each point (line) in the Radon transformation space is associated with the angle $a$ perpendicular to it, as well as the distance from the origin of point on the line which intersects the line $(x,y) = (t \cos(a), t \sin(y))$. (SOURCE: Natterer)



#### Adjoint

We can also consider the dual to the radon transform, $\mathcal{R}^*$ which sends each line of the function $\mathcal{R}f(r, \alpha)$ back to a point $x \in \mathbb{R}^n$. If $g(\gamma) = \mathcal{R}f(x)$, then 

\begin{align*} \mathcal{R}^* g (x) = \int_{x \in \gamma} g(\gamma) d\mu(\gamma), \end{align*}

where $d\mu(\gamma)$ is the "unique probability measure on $\{\gamma | x \in \gamma\}$". (SOURCE: WIKIPEDIA)

Later on we might define more explicitly what this measure is, and use it to define the Radon transform on a generalized function $T$, by using a test function $g$ and duality: $\langle \mathcal{R}T , g \rangle = \langle T, \mathcal{R}^* g \rangle$.

In the case of $\mathbb{R}^2$, the dual radon transform is 

\begin{align*}  \end{align*}




## 1.3 Filtered Back Projection

#### Issues with simple back projection

Consider 

\begin{align*} \mathcal{R}^*(\mathcal{R}f(a)) = \end{align*}

Simply applying the adjoint radon transform to a sinogram fails to invert the Radon transform. The projection data is simply smeared back accross the spatial domain, which results in blurring and star shaped artifacts. In effect, the back projection acts as a low pass filter (WHY IS THIS? WHAT IS THE FREQUENCY CUTOFF?) and obscures details of the image which correspond to high frequencies in the Fourier domain, such as edges and fine-grained patterns.  

#### Filtered Back Projection

To avoid this, a ramp filter $|\omega|$ is introduced, which amplifies higher frequencies in the fourier domain, and subdues the low-frequency blurring.

However, introduction of this high-frequency-amplifying ramp filter makes the filtered back projection extremely sensitive to high-frequency noise, and ill-posed. 

So we introduce additional filtering $|\omega|W(\omega)$, often in the form of a window which removes the highest frequencies, while still keeping the overall ramp shape. Note that $W(\omega)$ is a weight function, which approximates the identity in some sense. 

Examples of filters are Ram-lak, shepp-philips, Lam, and others. We will first investigate very basic filters which are not effective.

## 2. Theory of Radon transforms in $\mathbb{R}^2$

## 2.1 Fourier Slice theorem

#### Fourier inversion theorem

For a continuous function $f$, we define the Fourier transform 

\begin{align*} (\mathcal{F}f)(\xi) = \int_{-\infty}^{\infty} e^{-2i \pi y \xi} f(y) dy. \end{align*}

For functions $f$ satisfying certain conditions, and denoting $g(\xi) = \mathcal{F}f(\xi)$, the inverse Fourier transform is defined as

\begin{align*} (\mathcal{F}^{-1}g)(x) = \int_{-\infty}^{\infty} e^{2i \pi x \xi}g(\xi) d\xi. \end{align*}

(PROOF FOR SCHWARTZ FUNCTIONS?)

Taken together, the Fourier inversion theorem states that, for functions $f$ satisfying certain conditions, $f$ can be reconstruced from its fourier transform by applying the inverse fourier transform, that is, 

\begin{align*} f(x) &= \int_{-\infty}^{\infty} e^{2i \pi x \xi} \left( \int_{-\infty}^{\infty} e^{-2i \pi y \xi} f(y) dy \right)d\xi. \\
&=  \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} e^{2i \pi (x - y) \xi}f(y) dy d\xi. \end{align*}  

One such satisfactory condition would be that $f$ belongs to the Schwartz space.

## 2.2 Mapping of Schwartz space

For our purposes, we will assume that the function we wish to construct is rapidly decreasing. That is, it is in the Schwartz space 

\begin{align*} \mathcal{S}(\mathbb{R}^2) = \{f \in |  \} \end{align*}

## Inversion of Radon transform

#### Ludwig's inversion formula, Hilbert transform

#### Nievergelt's elementary inversion method

#### Riesz potential

# 3. Techniques in Parallel beam X-Ray tomography

## 3.1 Literature review, list of filters

#### Ram-Lakk, or Ramp filter

The Ram-Lakk filter is just the ramp filter with no transfer function, and is the starting point for all of the other filters. It is $|\omega|$, where $\omega$ is in the frequency domain.

## 3.2 Noise

The two types of noise that occur in X-ray imaging scenarios are poisson noise (quantum fuzz) and gaussian noise (photon/electron... noise?), equations () and () respectively.

\begin{align*}


\end{align*}

## 3.3 Numerical analysis

#### Nyquist, sampling

# 4. Learning of Filter 

## 4.1 Multilinear regression to train a difference-of-gaussians filter window $W(\omega)$

We will generate a dataset consisting of paired ellipse-phantoms and noised sinograms. We will use various mixtures of the two types of noise common to real-world CT applications, namely Poisson and Gaussian. The windowing function will be initialized as a sum of Gaussians with set means and variances. The paramters to be altered will be the weight of each Gaussian (allowing negative weights.) We will use least squares regression to optimize the performance of $W(\omega)$ as a filter window during FBP on the given dataset. (SEE filter_learning.ipynb FOR MORE DETAIL)

## 4.2 training a CNN $W(\theta; \omega, \mathcal{R}f)$

(SEE filter_learning.ipynb)

## 4.3 Training a noise-assessment neural network $A(\theta_1; \mathcal{R}f)$, and a filter-selector function $G(N)$ which selects a filter paramaterization $W_{N}(\omega)$ 

$G(N)$, which chooses an optimal filter based off known (approximate) noise coefficients $N$, can also be a neural network, which learns on a dataset of paired noise mixtures $N$ and optimal filter paramterizations $W_N$. Each $W_N$ can be learned using the multilinear regression on a dataset of ellipse-phantoms with the noising $N$, as in section 4.1. However, the MSE loss function should be replaced with a more sophisticated combination of error measures, so that there is no MSE-specific bias. 

(And in this instance we are abusing notation to have $W_N$ refer to both the difference-of-gaussians function $W_N(\omega)$, as well as the respective paramaterization (weighting of the Gaussians).)

(SEE filter_learning.ipynb)

# 5. A brief discussion of further topics

## 5.1 Radon transform in $\mathbb{R}^3$ and $\mathbb{R}^n$

## 5.3 Radon transform of Tempered Distributions

Dissributions are generalized functions, which are linear functionals acting on a space of test functions. For example, one family of distributions can be defined by taking the space of functions to be the functions with compact support, $D(\mathbb{R})$.

So distributions are linear operators $T: D(\mathbb{R}) \to \mathbb{R}$ which act on functions $\phi$. Instead of writing $T(\phi)$, we write $\langle T, \phi \rangle$ to denote the distribution $T$ evaluated at the test function $\phi$.

For example, the Dirac delta $\delta$ can be defined as 

\begin{align*} \langle \delta , \phi \rangle = \phi(0). \end{align*}

(SOURCE: Wikpiedia) "Derivatives of generalized functions are defined via transposition to the test functions using integration by parts

\begin{align*}  \langle f', \phi \rangle = \int_{-\infty}^{\infty} f' \phi dx
= \bigl[f(x)\phi(x) \bigr]_{-\infty}^{\infty} - \int_{-\infty}^{\infty} f' \phi dx
= - \langle f, \phi' \rangle,\end{align*} 

(Note that $\bigl[f(x)\phi(x) \bigr]_{-\infty}^{\infty} = 0$. Since $\phi \in D(\mathbb{R})$, it is equal to zero outside of some bounded interval.) 

allowing differentiation of non-differentiable or discontinuous functions (like the Heaviside step function or Dirac delta function)." 

The tempered distributions are the generalized functions which take the Schwartz space as their space of test functions. 

The theory we have been discussing for Radon transforms of functions in the schwartz space also holds for tempered distributions. 

#### Fourier inversion


